# TABLE - Phát hiện và xử lý mất cân bằng lớp: Áp dụng SMOTE, ADASYN và Random Under-sampling.

### Phát hiện và Xử lý Mất cân bằng lớp (Class Imbalance)

Trong các bài toán thực tế (như phát hiện gian lận, chẩn đoán y khoa, hay dự đoán hồ sơ giấy phép), dữ liệu hiếm khi phân bố đồng đều. Việc nhóm đa số (Majority Class) áp đảo nhóm thiểu số (Minority Class) sẽ khiến mô hình sinh ra thiên kiến (bias): Nó có xu hướng luôn dự đoán thuộc về nhóm đa số để tối ưu hóa độ chính xác (Accuracy), dẫn đến việc bỏ lọt hoàn toàn các trường hợp thiểu số quan trọng.

Để giải quyết vấn đề này, các kỹ thuật tái cân bằng dữ liệu (Resampling) được áp dụng. Tuy nhiên, việc áp dụng chúng đòi hỏi sự tuân thủ nghiêm ngặt về quy trình.

#### 1. Tại sao KHÔNG ĐƯỢC tái cân bằng trước khi chia tập Train/Test?
Việc áp dụng SMOTE, ADASYN hay bất kỳ kỹ thuật resampling nào *trước* khi chia dữ liệu thành tập huấn luyện (Train) và tập kiểm thử (Test) là một sai lầm nghiêm trọng, dẫn đến hiện tượng **Rò rỉ dữ liệu (Data Leakage)**.

* **Bản chất của sự rò rỉ:** Các thuật toán Over-sampling (như SMOTE/ADASYN) hoạt động bằng cách nội suy để sinh ra các điểm dữ liệu "ảo" nằm giữa các điểm dữ liệu "gốc" của nhóm thiểu số. Nếu ta sinh dữ liệu trước khi chia tập, một điểm "gốc" có thể lọt vào tập Train, trong khi điểm "ảo" (mang thông tin gần như giống hệt) lại lọt vào tập Test.
* **Hậu quả ảo tưởng:** Mô hình sẽ học thuộc lòng (memorize) điểm gốc trong lúc train, và dễ dàng "đoán trúng" điểm ảo khi test. Điều này đẩy các chỉ số đánh giá (Precision, Recall, F1) lên cao ngất ngưởng một cách giả tạo. Nhưng khi triển khai thực tế trên dữ liệu hoàn toàn mới, mô hình sẽ thất bại.
* **Quy trình chuẩn:** Tập Test phải là đại diện chân thực của thế giới thực (vốn dĩ mất cân bằng). Do đó, **bắt buộc phải chia Train/Test trước**, sau đó **chỉ được phép thực hiện Resampling trên tập Train**.

#### 2. Bản chất và So sánh các kỹ thuật Resampling
Nhóm đã tiến hành nghiên cứu và cài đặt 3 phương pháp tái cân bằng mang tính chất đại diện cho các trường phái khác nhau: Random Under-sampling (Giảm mẫu), SMOTE (Tăng mẫu tuyến tính), và ADASYN (Tăng mẫu thích ứng).

**A. Random Under-sampling (RUS)**
* **Bản chất:** Giảm số lượng mẫu của nhóm đa số bằng cách loại bỏ ngẫu nhiên các điểm dữ liệu cho đến khi số lượng bằng với nhóm thiểu số.
* **Ưu điểm:** Cực kỳ nhanh, làm giảm kích thước tập dữ liệu giúp tăng tốc độ huấn luyện mô hình.
* **Nhược điểm:** Mất mát thông tin (Information Loss). Việc xóa ngẫu nhiên có thể vô tình vứt bỏ những đặc trưng quan trọng của nhóm đa số, khiến mô hình bị Underfitting (chưa học tới).
* **Khi nào sử dụng:** Chỉ nên dùng khi tập dữ liệu có kích thước "khổng lồ" (hàng triệu dòng), năng lực tính toán bị giới hạn, và nhóm đa số có mức độ trùng lặp thông tin (redundancy) rất cao.

**B. SMOTE (Synthetic Minority Over-sampling Technique)**
* **Bản chất:** Tăng số lượng mẫu nhóm thiểu số bằng cách tạo ra dữ liệu nhân tạo. Thuật toán chọn một điểm thiểu số, tìm $k$ điểm lân cận gần nhất ($k$-Nearest Neighbors) của nó. Sau đó, nó vẽ các đường thẳng nối điểm gốc với các điểm lân cận và chọn ngẫu nhiên các điểm mới nằm trên các đoạn thẳng này.
* **Ưu điểm:** Không sao chép y hệt dữ liệu cũ (như Random Over-sampling), giúp giảm thiểu nguy cơ Overfitting. Mô hình học được các vùng đặc trưng rộng hơn của nhóm thiểu số.
* **Nhược điểm:** SMOTE sinh dữ liệu một cách "mù quáng". Nếu một điểm thiểu số đang nằm sâu trong vùng của nhóm đa số (nhiễu), SMOTE vẫn sẽ tạo ra thêm các điểm xung quanh nó, làm tăng độ nhiễu và khiến ranh giới quyết định (Decision Boundary) bị mờ nhạt.
* **Khi nào sử dụng:** Phù hợp với các tập dữ liệu mà các lớp đã có sự tách biệt tương đối rõ ràng, và nhóm thiểu số cần được làm "đậm" thêm để thuật toán chú ý.

**C. ADASYN (Adaptive Synthetic Sampling)**
* **Bản chất:** Là phiên bản nâng cấp thông minh của SMOTE. ADASYN cũng dùng $k$-NN để sinh dữ liệu nội suy, nhưng nó tính toán thêm một **trọng số (density distribution)**. Điểm thiểu số nào càng bị bao vây bởi nhiều điểm đa số (tức là càng "khó" để mô hình phân loại) thì ADASYN càng sinh ra nhiều điểm nhân tạo quanh nó.
* **Ưu điểm:** Tự động điều chỉnh trọng tâm học tập của mô hình vào những ranh giới phân loại khó khăn nhất (Hard-to-learn examples).
* **Nhược điểm:** Rất nhạy cảm với ngoại lai (Outliers). Nếu một điểm thiểu số là dữ liệu rác/nhiễu lạc lõng giữa bầy đa số, ADASYN sẽ sinh ra một đống dữ liệu ảo quanh cục nhiễu đó, làm hỏng hoàn toàn mô hình.
* **Khi nào sử dụng:** Rất mạnh mẽ khi ranh giới giữa nhóm đa số và thiểu số phức tạp, chồng lấn lên nhau (Highly non-linear boundary), miễn là dữ liệu đã được dọn sạch ngoại lai kỹ càng từ bước trước.


In [1]:
from table.dataset import TableDataset
from config.settings import PATH_FOLDER_TABLE_RAW
from pathlib import Path
dataset = TableDataset(str(Path(PATH_FOLDER_TABLE_RAW) / "Building_Permits.csv"))

In [2]:
import pandas as pd
import numpy as np
from table.imbalance_handler import ImbalanceHandler 

# Chọn cột 'Permit Type' làm biến mục tiêu (Target - y)
target_col = 'Permit Type' 

# Giả sử df_clean là dataframe của bạn sau khi đã xử lý Missing Value và Outlier
df_clean = dataset.data.dropna(subset=[target_col]).copy()

# Chọn ra các features (X) hợp lệ (Chỉ lấy các cột số đã được chuẩn hóa)
# Bỏ cột target và các cột ID vô nghĩa
ignore_for_x = [target_col, 'Record ID', 'Permit Number', 'Zipcode', 'Block', 'Lot']
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in numeric_cols if c not in ignore_for_x]

# Nếu dữ liệu vẫn còn NaN, điền tạm bằng Median để thuật toán SMOTE chạy được
X = df_clean[feature_cols].fillna(df_clean[feature_cols].median())
y = df_clean[target_col]

print(f"Kích thước tập X: {X.shape}")
print("Phân phối của cột Target (Permit Type):")
print(y.value_counts(normalize=True) * 100) # In ra % để thấy rõ sự mất cân bằng

# 2. CHẠY CLASS XỬ LÝ MẤT CÂN BẰNG
# Nếu data quá lớn (> 100k dòng), test_size = 0.2, thuật toán SMOTE có thể chạy hơi lâu (1-2 phút)
bonus_task = ImbalanceHandler(X=X, y=y, test_size=0.2, random_state=42)
bonus_task.run()

Kích thước tập X: (198900, 12)
Phân phối của cột Target (Permit Type):
Permit Type
8    89.916541
3     7.372046
4     1.453997
2     0.477627
6     0.301659
7     0.256913
1     0.175465
5     0.045752
Name: proportion, dtype: float64
Phân phối nhãn gốc:
Permit Type
8    178844
3     14663
4      2892
2       950
6       600
7       511
1       349
5        91
Name: count, dtype: int64

Đang chạy thử nghiệm các chiến lược Resampling...


KeyboardInterrupt: 